# WeatherGraph — Research Playbook

Run global weather forecasts with the C++/ONNX backend from any Jupyter
environment (Colab, Kaggle, local). The engine accepts **any ONNX model** —
see the model preparation section for PyTorch, JAX, and TensorFlow.

---

**Input contract** — the engine expects `float32[1, 71042, 78]`:
- **71042 nodes** — 1° ERA5 grid (181 × 360) merged with H3 mesh
- **78 channels** — variables `[z, q, t, u, v, w]` at 13 pressure levels
  (50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000 hPa)

In [ ]:
# ── Cell 1: Environment setup ─────────────────────────────────────────────────
# Run once per Colab/Kaggle session. Skip locally if you have a pre-built wheel.
import sys, subprocess, os

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB or IN_KAGGLE:
    # System build tools
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'cmake', 'g++'], check=True)

    # Clone the engine repo
    if not os.path.exists('/tmp/weathergraph'):
        subprocess.run([
            'git', 'clone', '--depth=1',
            'https://github.com/Wanderspool/WeatherGraph',
            '/tmp/weathergraph',
        ], check=True)

    os.chdir('/tmp/weathergraph')

    # Download ONNX Runtime shared library
    subprocess.run(['make', 'onnxruntime'], check=True)

    # Install build deps and build the Python extension
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'pybind11>=2.13.0', 'scikit-build-core',
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--no-build-isolation', '.',
    ], check=True)

    # Pre-load the shared library so it is discoverable at import time
    import ctypes
    for lib in sorted(__import__('glob').glob('weathergraph/core/libonnxruntime.so*')):
        ctypes.CDLL(lib)

print('Engine environment ready.')

---
## Step 1 — Prepare your model

Run **exactly one** of the cells below, then proceed to Step 2.

In [ ]:
# ── Option A: Download a pre-built ONNX (HuggingFace or any HTTPS URL) ───────
import urllib.request, os

MODEL_PATH = 'models/weather_gnn.onnx'
os.makedirs('models', exist_ok=True)

# Default: reference 2022 ONNX artifact on HuggingFace.
MODEL_URL = 'https://huggingface.co/Wanderspool/Keisler_2022/resolve/main/keisler_2022.onnx'
# Other examples:
#   MODEL_URL = 'https://my-bucket.s3.amazonaws.com/models/graphcast.onnx'
#   MODEL_URL = 'https://github.com/MyOrg/Repo/releases/download/v1.0/model.onnx'

urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print(f'Downloaded {os.path.getsize(MODEL_PATH) / 1e6:.1f} MB  →  {MODEL_PATH}')

In [ ]:
# ── Option B: Convert from PyTorch ───────────────────────────────────────────
# pip install torch  — run if torch is not already installed
import torch, os

MODEL_PATH = 'models/my_pytorch_model.onnx'
os.makedirs('models', exist_ok=True)

# ── Replace with your model ──────────────────────────────────────────────────
# from my_package import MyWeatherGNN
# model = MyWeatherGNN()
# model.load_state_dict(torch.load('my_weights.pt', map_location='cpu'))
# model.eval()
# ─────────────────────────────────────────────────────────────────────────────
# Demo: use a trivial identity module as a stand-in
model = torch.nn.Identity()

dummy_input = torch.zeros(1, 71042, 78, dtype=torch.float32)
torch.onnx.export(
    model, dummy_input, MODEL_PATH,
    opset_version=17,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
)
print(f'Exported {os.path.getsize(MODEL_PATH) / 1e6:.1f} MB  →  {MODEL_PATH}')

In [ ]:
# ── Option C: Convert from TensorFlow / Keras (tf2onnx) ──────────────────────
# pip install tensorflow tf2onnx  — run if not already installed
import subprocess, sys, os

SAVED_MODEL_DIR = 'my_saved_model'   # ← your tf.saved_model directory
MODEL_PATH      = 'models/my_tf_model.onnx'
os.makedirs('models', exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'tf2onnx.convert',
    '--saved-model', SAVED_MODEL_DIR,
    '--output',      MODEL_PATH,
    '--opset',       '17',
], check=True)
print(f'Converted  →  {MODEL_PATH}')

In [ ]:
# ── Option D: Build a WeatherGraph-compatible ONNX from repo scripts (requires LFS) ──
# This uses exporter/build_gnn_graph.py which reads data/weights/ and
# data/graph_data/ — both are Git LFS objects, already present in the repo.
import subprocess, sys, os

MODEL_PATH = 'models/weather_gnn.onnx'
os.makedirs('models', exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', 'numpy', 'onnx', 'onnxscript'
], check=True)
subprocess.run([
    sys.executable, 'exporter/build_gnn_graph.py', '--output', MODEL_PATH
], check=True)
print(f'Built  →  {MODEL_PATH}')

In [ ]:
# ── Validate ONNX (always run after choosing a model option above) ────────────
import onnx

model_proto = onnx.load(MODEL_PATH)
onnx.checker.check_model(model_proto)
opset = model_proto.opset_import[0].version
print(f'Model OK  — IR v{model_proto.ir_version}, opset {opset}')
print(f'Path: {MODEL_PATH}')

---
## Step 2 — Select your data source

The engine accepts **any** data source that can produce the required
`float32[1, 71042, 78]` input tensor.  Use one of the built-in adapters
or build a custom one with the constructor.

| Adapter | Auth required? | Extra install |
|---|---|---|
| `era5_netcdf` | — | — |
| `ecmwf_open` | — | `pip install ecmwf-opendata cfgrib eccodes` |
| `cds_era5` | API key (~/.cdsapirc) | `pip install cdsapi` |
| `gfs` | — | `pip install herbie-data` |
| `open_meteo` | — | — |
| `custom` | — | depends on format |

Run **exactly one** of the cells below, then proceed to Step 3.

In [ ]:
# ── Show all available data sources ──────────────────────────────────────────
from weathergraph.data_sources import list_sources
list_sources()

In [ ]:
# ── Option 1: ERA5 local NetCDF (default, no extra deps) ─────────────────────
# Download ERA5 pressure-level data from https://cds.climate.copernicus.eu/
# and point this adapter at the file.

from weathergraph.data_sources import load_source

DATA_ADAPTER = load_source("era5_netcdf", path="data/era5_archives/init_state.nc")

# Inspect the dataset:
# ds = DATA_ADAPTER.load()
# print(ds)
print(f'Adapter ready: {DATA_ADAPTER.name}')

In [ ]:
# ── Option 2: ECMWF Open Data — free, no API key ─────────────────────────────
# pip install ecmwf-opendata cfgrib eccodes
#
# Downloads the latest ECMWF IFS forecast from the public HTTPS endpoint.
# Updated twice daily (00Z and 12Z runs).

from weathergraph.data_sources import load_source

DATA_ADAPTER = load_source(
    "ecmwf_open",
    date=None,    # None = today; or "YYYY-MM-DD"
    time=0,       # model run: 0 or 12 (UTC hours)
    step=0,       # forecast lead time in hours (0 = analysis)
    target="/tmp/ecmwf_open.grib2",
)
print(f'Adapter ready: {DATA_ADAPTER.name}')

In [ ]:
# ── Option 3: Copernicus CDS ERA5 reanalysis — free registration + API key ───
# pip install cdsapi
#
# Create a free account at https://cds.climate.copernicus.eu/
# Place your credentials in ~/.cdsapirc:
#   url: https://cds.climate.copernicus.eu/api
#   key: <your-api-key>
#
# Or set environment variables CDSAPI_URL and CDSAPI_KEY.

from weathergraph.data_sources import load_source

DATA_ADAPTER = load_source(
    "cds_era5",
    date="2024-01-01",    # YYYY-MM-DD
    time="00:00",         # HH:00
    target="/tmp/cds_era5.nc",
)
print(f'Adapter ready: {DATA_ADAPTER.name}')

In [ ]:
# ── Option 4: NOAA GFS via AWS Open Data — free, no key ──────────────────────
# pip install herbie-data
#
# Downloads GRIB2 slices from AWS S3 Open Data.
# Available runs: 00Z, 06Z, 12Z, 18Z.  Latency: ~4 h after run time.

from weathergraph.data_sources import load_source

DATA_ADAPTER = load_source(
    "gfs",
    date="2024-01-01 00:00",   # "YYYY-MM-DD HH:MM"
    fxx=0,                      # forecast hour (0 = analysis)
    source="aws",               # "aws" | "nomads" | "google" | "azure"
)
print(f'Adapter ready: {DATA_ADAPTER.name}')

In [ ]:
# ── Option 5: Open-Meteo — free multi-model API, no key needed ───────────────
# Single-point forecast.  Useful for quick tests and point-scale validation.
# For global grid runs, use ERA5 or GFS instead.

from weathergraph.data_sources import load_source

DATA_ADAPTER = load_source(
    "open_meteo",
    latitude=51.5,       # degrees N
    longitude=-0.1,      # degrees E  (negative = W)
    forecast_days=10,
    model="best_match",  # "ecmwf_ifs025" | "gfs_seamless" | "icon_seamless"
)
print(f'Adapter ready: {DATA_ADAPTER.name}')

In [ ]:
# ── Option 6: Custom adapter — any file format, any variable names ───────────
#
# Supports NetCDF4, GRIB2 (cfgrib), Zarr, HDF5, and raw NumPy arrays.
# Use variable_map to translate your field names to the engine contract.

from weathergraph.data_sources import CustomAdapter

# ── A: Local NetCDF with non-standard variable names ─────────────────────────
DATA_ADAPTER = CustomAdapter(
    path="my_nwp_output.nc",
    format="netcdf4",                # "netcdf4" | "grib2" | "zarr" | "hdf5"
    variable_map={
        "z": "geopotential",         # model_name: source_name
        "q": "specific_humidity",
        "t": "air_temperature",
        "u": "eastward_wind",
        "v": "northward_wind",
        "w": "vertical_velocity",
    },
    level_dim="pressure",            # dimension name for pressure levels
    lat_dim="lat",                   # dimension name for latitude
    lon_dim="lon",                   # dimension name for longitude
    level_unit="hPa",                # "hPa" | "Pa"  (auto-converts if "Pa")
    geopot_in_meters=False,          # True  → multiply z by g=9.80665
)

# ── B: GRIB2 file  (pip install cfgrib eccodes) ───────────────────────────────
# DATA_ADAPTER = CustomAdapter(
#     path="icon_forecast.grb2",
#     format="grib2",
#     variable_map={"z": "z_height", "t": "t2"},
#     geopot_in_meters=True,
# )

# ── C: Pre-built NumPy array  [1, 71042, 78] ─────────────────────────────────
# import numpy as np
# raw = np.load("my_state.npy").astype(np.float32)
# DATA_ADAPTER = CustomAdapter(data=raw)

# ── D: From a dict schema (config-file driven pipelines) ─────────────────────
# DATA_ADAPTER = CustomAdapter.from_schema({
#     "source":       "forecast.nc",
#     "variable_map": {"z": "geopotential", "t": "temperature"},
#     "level_dim":    "pressure",
# })

print(f'Adapter ready: {DATA_ADAPTER.name}')

In [ ]:
# ── Validate: load one time step and inspect ──────────────────────────────────
# Run this after choosing any adapter above.
# NOTE: this will trigger the actual download / file-read.

try:
    ds = DATA_ADAPTER.load()
    print('Dataset loaded successfully.')
    print(ds)
except (FileNotFoundError, ImportError) as e:
    print(f'[SKIP] {e}')
    print('Provide a real file path or configure your API key, then re-run.')
    ds = None

---
## Step 3 — Run inference

In [ ]:
# ── Single 6-hour step ────────────────────────────────────────────────────────
import numpy as np
import weathergraph_backend   # the C++ extension module

engine = weathergraph_backend.WeatherGraphEngine(MODEL_PATH)

# Replace with your real ERA5 initial state [1, 71042, 78] float32
state = np.random.randn(1, 71042, 78).astype(np.float32)

prediction = engine.predict(state)
print(f'Output shape : {prediction.shape}')          # (1, 71042, 78)
print(f'Max abs value: {np.abs(prediction).max():.4f}')

In [ ]:
# ── 10-day autoregressive rollout (40 × 6 h steps) ───────────────────────────
STEPS = 40

current    = state.copy()
trajectory = [current]

for step in range(STEPS):
    current = engine.predict(current)
    trajectory.append(current.copy())
    if (step + 1) % 8 == 0:
        print(f'  step {step+1:3d}/40  ({(step+1)*6}h)  '
              f'mean={current.mean():.4f}  std={current.std():.4f}')

trajectory = np.stack(trajectory, axis=0)   # (41, 1, 71042, 78)
print(f'\nFull trajectory shape: {trajectory.shape}')

In [ ]:
# ── High-level xarray API (requires real data) ──────────────────────────────
# Uncomment and fill in your data paths.
#
# from weathergraph import WeatherGraphModel
# from weathergraph.data_sources import load_source
#
# model = WeatherGraphModel(model_path=MODEL_PATH, weights_dir='data')
#
# ── Option A: pass an adapter directly ───────────────────────────────────────
# adapter = load_source('era5_netcdf', path='path/to/era5_initial_state.nc')
# forecast = model.forecast(adapter, steps=40)
#
# ── Option B: pass a pre-loaded xarray.Dataset ───────────────────────────────
# import xarray as xr
# ds = xr.open_dataset('path/to/era5_initial_state.nc')
# forecast = model.forecast(ds, steps=40)
#
# ── Option C: use DATA_ADAPTER from Step 2 ───────────────────────────────────
# forecast = model.forecast(DATA_ADAPTER, steps=40)

print('xarray API ready — uncomment above and choose a data source in Step 2.')